# Lecture 27 - Feature Engineering and Preprocessing

## Learning Objectives

- Apply one-hot encoding with OneHotEncoder and pd.get_dummies()
- Distinguish label encoding from ordinal encoding
- Standardise features with StandardScaler
- Normalise features with MinMaxScaler
- Use ColumnTransformer to mix column types
- Chain preprocessing + modelling with Pipeline
- Generate polynomial features for non-linear relationships

## Key Topics

- OneHotEncoder vs pd.get_dummies()
- Label encoding vs ordinal encoding
- StandardScaler
- MinMaxScaler
- ColumnTransformer
- Pipeline
- PolynomialFeatures

## One-Hot Encoding with OneHotEncoder and pd.get_dummies()

Most machine learning algorithms require numerical input. Categorical variables (like "red", "blue", "green") must be converted into numbers. **One-hot encoding** creates a binary column for each category: for a feature with k categories, it produces k binary columns, exactly one of which is 1 for each row.

Pandas provides `pd.get_dummies()` for quick one-hot encoding on DataFrames. Scikit-learn's `OneHotEncoder` integrates seamlessly with Pipelines and `ColumnTransformer`, making it the preferred choice for production workflows.

Be aware of the **dummy variable trap**: when a categorical feature has k categories, you only need k-1 dummy columns to avoid perfect multicollinearity. Use `drop="first"` in OneHotEncoder or `drop_first=True` in pd.get_dummies() to handle this.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Sample data with mixed types
df = pd.DataFrame({
    "color": ["red", "blue", "green", "blue", "red"],
    "size": ["S", "M", "L", "M", "XL"],
    "price": [10, 15, 20, 12, 18]
})
print("Original DataFrame:")
print(df)

# One-hot encoding with pandas
df_dummies = pd.get_dummies(df, columns=["color", "size"], drop_first=False)
print("\nAfter pd.get_dummies():")
print(df_dummies)

In [ ]:
# One-hot encoding with sklearn OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop="first")
encoded = ohe.fit_transform(df[["color", "size"]])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
print("After OneHotEncoder:")
print(encoded_df)

## Label Encoding vs Ordinal Encoding

**Label encoding** assigns each category a unique integer (0, 1, 2, ...). This is simple but can mislead models into thinking categories have an ordinal relationship (e.g., 0 < 1 < 2). For nominal categories like colours, this is dangerous.

**Ordinal encoding** is for variables where the categories do have a natural order (e.g., "small" < "medium" < "large"). You assign integers that respect this ordering. Scikit-learn provides `OrdinalEncoder` with an explicit `categories` parameter to define the order.

Rule of thumb: use one-hot encoding for nominal categories (no order) and ordinal encoding for categories with a clear ordering.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# OrdinalEncoder for features with a natural order
sizes = pd.DataFrame({"size": ["S", "M", "L", "XL", "M", "S"]})
ord_enc = OrdinalEncoder(categories=[["S", "M", "L", "XL"]])
sizes_encoded = ord_enc.fit_transform(sizes)
print("Ordinal encoding of sizes:")
print(pd.DataFrame({"size_original": sizes["size"],
                     "size_encoded": sizes_encoded.flatten().astype(int)}))

# LabelEncoder is for the target variable (y), not features
le = LabelEncoder()
y_labels = le.fit_transform(["cat", "dog", "bird", "dog", "cat"])
print("\nLabel encoding of target:", y_labels)

## Standardization and Normalization

Many ML algorithms (SVM, KNN, LogisticRegression with regularisation, PCA) assume features have similar scales. If one feature ranges from 0-1 and another from 0-100000, the latter dominates the distance calculations.

- **StandardScaler** standardises features by removing the mean and scaling to unit variance: z = (x - μ) / σ. The result has mean 0 and standard deviation 1. Best for algorithms that assume normally distributed data.
- **MinMaxScaler** scales features to a fixed range (usually [0, 1]): x_scaled = (x - min) / (max - min). Preserves the shape of the original distribution. Best for neural networks and algorithms that don't assume any distribution.

Always **fit** the scaler on the training set only, then **transform** both train and test sets to avoid data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

np.random.seed(42)
data = np.random.exponential(scale=2, size=(100, 3))

std_scaler = StandardScaler()
data_std = std_scaler.fit_transform(data)
print(f"StandardScaler - mean: {data_std.mean(axis=0).round(3)}")
print(f"StandardScaler - std:  {data_std.std(axis=0).round(3)}")

mm_scaler = MinMaxScaler()
data_mm = mm_scaler.fit_transform(data)
print(f"\nMinMaxScaler - min: {data_mm.min(axis=0).round(3)}")
print(f"MinMaxScaler - max: {data_mm.max(axis=0).round(3)}")

## ColumnTransformer for Mixed Column Types

Real-world datasets contain a mix of numeric, categorical, and sometimes text features. Applying different preprocessing steps to different columns by hand is error-prone and leads to messy code.

`ColumnTransformer` applies different transformers to different columns in a single object. You specify a list of (name, transformer, columns) tuples. It handles the bookkeeping of joining the transformed columns back together.

This is the professional way to preprocess heterogeneous data in scikit-learn.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

df_full = pd.DataFrame({
    "age": [25, 45, 30, 50, 35],
    "income": [50000, 80000, 60000, 120000, 75000],
    "education": ["HS", "College", "Graduate", "College", "Graduate"],
    "city": ["NYC", "LA", "SF", "NYC", "LA"]
})

ct = ColumnTransformer([
    ("num", StandardScaler(), ["age", "income"]),
    ("cat", OneHotEncoder(drop="first"), ["education", "city"])
])

transformed = ct.fit_transform(df_full)
print("Transformed shape:", transformed.shape)
print(pd.DataFrame(transformed, columns=ct.get_feature_names_out()))

## Pipeline: Chaining Preprocessing + Model

A `Pipeline` chains multiple transformers and a final estimator into a single object. When you call `.fit()`, each step transforms the data sequentially before passing it to the final model. During `.predict()`, the same transformations are applied automatically.

Pipelines ensure that preprocessing is applied consistently to train and test sets, prevent data leakage, and make your code cleaner and more reproducible. They are essential for proper cross-validation: `cross_val_score(pipeline, X, y)` will correctly fit the scaler on each training fold separately, avoiding any contamination from the validation fold.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=50, random_state=42))
])

# Using the same df_full: create a target
y_full = pd.Series([0, 1, 0, 1, 1], name="high_income")
X_full = df_full

pipe.fit(X_full, y_full)
print("Pipeline score:", pipe.score(X_full, y_full))

In [ ]:
# Cross-validation with pipeline (no data leakage!)
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipe, X_full, y_full, cv=3)
print(f"CV scores: {scores}")
print(f"Mean CV accuracy: {scores.mean():.3f}")

## PolynomialFeatures and Interaction Terms

Linear models can only capture linear relationships. When the relationship between features and target is curved, you need **polynomial features**: adding squared or cubed versions of existing features (x → x, x², x³) lets a linear model fit non-linear patterns.

`PolynomialFeatures` also generates **interaction terms** (x₁ × x₂). These capture multiplicative effects — for example, the combined effect of age and exercise on health.

Be cautious: the number of features grows quickly. A dataset with 10 features and degree 3 produces hundreds of features, risking overfitting and slower computation.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

X_simple = np.arange(1, 11).reshape(-1, 1)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X_simple)

print("Original -> Polynomial (deg=3):")
print(pd.DataFrame(X_poly, columns=poly.get_feature_names_out(["x"])))

In [ ]:
# Pipeline with polynomial features + regression
from sklearn.linear_model import LinearRegression

poly_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

np.random.seed(42)
x = np.linspace(-3, 3, 100).reshape(-1, 1)
y = x.ravel()**3 - 2 * x.ravel()**2 + np.random.normal(0, 2, 100)

poly_pipe.fit(x, y)
print("Polynomial regression R²:", poly_pipe.score(x, y))

## Data Science Connection

Feature engineering is where domain expertise meets data science. Raw data rarely comes in a form ready for modelling — you must encode categories, scale numbers, create interaction terms, and build automated preprocessing pipelines. These skills separate a good data scientist from a great one. The ColumnTransformer and Pipeline tools you learned here are used daily at companies like Netflix (recommendation features), Uber (pricing features), and in virtually every Kaggle-winning solution.